In [ ]:
# ============================================================
# Hybrid LSTM-GARCH — Step 5: Hybrid Evaluation and Comparison
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Load all outputs ──────────────────────────────────────────
garch_df   = pd.read_csv('data/garch_outputs.csv',
                         index_col='Date', parse_dates=True)
lstm_preds = pd.read_csv('data/lstm_predictions.csv',
                         index_col='Date', parse_dates=True)

# ── Align all predictions on the test period ─────────────────
test_start = lstm_preds.index[0]
test_end   = lstm_preds.index[-1]

garch_test     = garch_df.loc[test_start:test_end].copy()
garch_forecast = garch_test['Cond_Vol'].shift(1).dropna()

common_idx = lstm_preds.index.intersection(garch_forecast.index)
y_true     = lstm_preds.loc[common_idx, 'True_Vol'].values
y_lstm     = lstm_preds.loc[common_idx, 'LSTM_Pred'].values
y_garch    = garch_forecast.loc[common_idx].values
y_hybrid   = (y_lstm + y_garch) / 2

models = {
    'GARCH(1,1)':        y_garch,
    'LSTM':              y_lstm,
    'Hybrid LSTM-GARCH': y_hybrid
}

# ── Overall metrics ───────────────────────────────────────────
print("=== Model Comparison — Test Period ===")
print(f"Test period: {common_idx[0].date()} to {common_idx[-1].date()}")
print(f"Test observations: {len(common_idx):,}\n")
print(f"{'Model':<22} {'RMSE':>10} {'MAE':>10} {'MAPE':>10} {'Dir Acc':>10}")
print("-" * 65)

results = {}
for name, y_pred in models.items():
    rmse    = np.sqrt(mean_squared_error(y_true, y_pred))
    mae     = mean_absolute_error(y_true, y_pred)
    mape    = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    dir_acc = np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred)))
    results[name] = {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'DirAcc': dir_acc}
    print(f"{name:<22} {rmse:>10.6f} {mae:>10.6f} {mape:>9.2f}% {dir_acc:>9.2%}")

In [ ]:
# ── Annualised metrics ────────────────────────────────────────
print("\n=== Annualised Metrics ===")
print(f"{'Model':<22} {'Ann.RMSE':>12} {'Ann.MAE':>12}")
print("-" * 48)
for name, r in results.items():
    print(f"{name:<22} {r['RMSE']*np.sqrt(252)*100:>11.4f}% {r['MAE']*np.sqrt(252)*100:>11.4f}%")

# ── RMSE improvement ──────────────────────────────────────────
garch_rmse  = results['GARCH(1,1)']['RMSE']
lstm_rmse   = results['LSTM']['RMSE']
hybrid_rmse = results['Hybrid LSTM-GARCH']['RMSE']

print("\n=== RMSE Improvement Over GARCH Baseline ===")
print(f"LSTM vs GARCH:    {(garch_rmse - lstm_rmse)/garch_rmse*100:+.2f}%")
print(f"Hybrid vs GARCH:  {(garch_rmse - hybrid_rmse)/garch_rmse*100:+.2f}%")
print(f"Hybrid vs LSTM:   {(lstm_rmse - hybrid_rmse)/lstm_rmse*100:+.2f}%")


In [ ]:
# ── Regime analysis ───────────────────────────────────────────
vol_33 = np.percentile(y_true, 33)
vol_67 = np.percentile(y_true, 67)

low_mask  = y_true <= vol_33
mid_mask  = (y_true > vol_33) & (y_true <= vol_67)
high_mask = y_true > vol_67

print(f"\n=== Performance by Volatility Regime ===")
print(f"Low vol  (< {vol_33*100*np.sqrt(252):.1f}% ann):   {low_mask.sum()} days")
print(f"Mid vol  ({vol_33*100*np.sqrt(252):.1f}–{vol_67*100*np.sqrt(252):.1f}% ann): {mid_mask.sum()} days")
print(f"High vol (> {vol_67*100*np.sqrt(252):.1f}% ann):  {high_mask.sum()} days\n")

for mask, label in [(low_mask, 'Low volatility'),
                    (mid_mask, 'Mid volatility'),
                    (high_mask, 'High volatility (crisis)')]:
    print(f"--- {label} ---")
    for name, y_pred in models.items():
        rmse = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
        print(f"  {name:<22}: RMSE = {rmse:.6f}")
    print()

# ── Save results ──────────────────────────────────────────────
pd.DataFrame(results).T.to_csv('data/model_comparison.csv')
print("Results saved to data/model_comparison.csv")

In [ ]:
# ── Plot 08: Model comparison ─────────────────────────────────
ann = lambda x: x * 100 * np.sqrt(252)
fig, axes = plt.subplots(3, 1, figsize=(14, 13))

# Panel 1: Full test period
axes[0].plot(common_idx, ann(y_true),
             color='#2166ac', linewidth=0.9, alpha=0.9,
             label='Actual volatility', zorder=3)
axes[0].plot(common_idx, ann(y_garch),
             color='#4d9221', linewidth=0.8, alpha=0.7,
             label='GARCH(1,1)', linestyle='--')
axes[0].plot(common_idx, ann(y_lstm),
             color='#d6604d', linewidth=0.8, alpha=0.7,
             label='LSTM')
axes[0].plot(common_idx, ann(y_hybrid),
             color='#762a83', linewidth=0.9, alpha=0.9,
             label='Hybrid LSTM-GARCH')
axes[0].set_title('Volatility Forecasts — Full Test Period (2018–2024)',
                  fontweight='bold', fontsize=12)
axes[0].set_ylabel('Annualised Volatility (%)')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Panel 2: COVID zoom
covid_mask = (common_idx >= pd.Timestamp('2020-01-01')) & \
             (common_idx <= pd.Timestamp('2020-09-30'))
covid_idx  = common_idx[covid_mask]

axes[1].plot(covid_idx, ann(y_true[covid_mask]),
             color='#2166ac', linewidth=1.2, label='Actual', zorder=3)
axes[1].plot(covid_idx, ann(y_garch[covid_mask]),
             color='#4d9221', linewidth=1.0, linestyle='--', label='GARCH(1,1)')
axes[1].plot(covid_idx, ann(y_lstm[covid_mask]),
             color='#d6604d', linewidth=1.0, label='LSTM')
axes[1].plot(covid_idx, ann(y_hybrid[covid_mask]),
             color='#762a83', linewidth=1.1, label='Hybrid')
axes[1].set_title('COVID Crisis Zoom (Jan–Sep 2020) — '
                  'How Each Model Handled the Spike',
                  fontweight='bold', fontsize=12)
axes[1].set_ylabel('Annualised Volatility (%)')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

# Panel 3: Bar chart
model_names = list(results.keys())
rmse_vals   = [results[m]['RMSE'] * np.sqrt(252) * 100 for m in model_names]
mae_vals    = [results[m]['MAE']  * np.sqrt(252) * 100 for m in model_names]
dir_vals    = [results[m]['DirAcc'] * 100 for m in model_names]
x           = np.arange(len(model_names))
width       = 0.25
colors      = ['#4d9221', '#d6604d', '#762a83']

bars1 = axes[2].bar(x - width, rmse_vals, width,
                    label='Ann. RMSE (%)', color=colors, alpha=0.85)
bars2 = axes[2].bar(x,          mae_vals, width,
                    label='Ann. MAE (%)',  color=colors, alpha=0.55)
axes[2].set_xticks(x)
axes[2].set_xticklabels(model_names, fontsize=10)
axes[2].set_title('Model Comparison — Annualised Error Metrics',
                  fontweight='bold', fontsize=12)
axes[2].set_ylabel('Error (%)')
axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3, axis='y')

for bar in bars1:
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.02,
                 f'{bar.get_height():.2f}%',
                 ha='center', va='bottom', fontsize=8)
for bar in bars2:
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.02,
                 f'{bar.get_height():.2f}%',
                 ha='center', va='bottom', fontsize=8)
for i, da in enumerate(dir_vals):
    axes[2].text(x[i] + width,
                 max(rmse_vals[i], mae_vals[i]) + 0.1,
                 f'Dir: {da:.1f}%',
                 ha='center', fontsize=8, color='#1a1a1a')

plt.tight_layout()
plt.savefig('outputs/08_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/08_model_comparison.png")

In [ ]:
# ── Plot 09: Error distributions ──────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, y_pred), color in zip(
    axes2,
    [('GARCH(1,1)', y_garch),
     ('LSTM', y_lstm),
     ('Hybrid LSTM-GARCH', y_hybrid)],
    ['#4d9221', '#d6604d', '#762a83']
):
    errors   = y_true - y_pred
    err_ann  = errors * 100 * np.sqrt(252)
    mean_err = np.mean(err_ann)
    ax.hist(err_ann, bins=60, color=color, alpha=0.7, edgecolor='none')
    ax.axvline(0, color='black', linewidth=1, linestyle='--')
    ax.axvline(mean_err, color='red', linewidth=1.5,
               label=f'Mean: {mean_err:.3f}%')
    ax.set_title(f'{name}\nError Distribution',
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('Forecast Error (annualised %)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig2.suptitle('Forecast Error Distributions — GARCH vs LSTM vs Hybrid',
              fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/09_error_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/09_error_distributions.png")

print("\n=== PROJECT COMPLETE ===")
print("Outputs saved:")
print("  data/model_comparison.csv")
print("  outputs/01 through 09 plots")